# Model calibration

### Set up


In [ ]:
# load the required libraries
using DataFrames
using CSV
using Statistics
using JuMP
using Gurobi

project_root = dirname(@__DIR__)

# load the function that models the electricity market and the other auxiliary functions
include(joinpath(project_root, "scripts", "model_electricity_market.jl"))
include(joinpath(project_root, "scripts", "auxiliar_functions.jl"))

# load the fixed datasets
historical_data        = CSV.read(joinpath(project_root, "data", "historical_data.csv"), DataFrame)
technology_data        = CSV.read(joinpath(project_root, "data", "technology_data.csv"), DataFrame)
projection_deltas_data = CSV.read(joinpath(project_root, "data", "projection_deltas_data.csv"), DataFrame)

### Running the model

Instead of projecting the historical dataset to the future as in the MC loop, we just run it once to 

In [113]:
baseline_years = [2023, 2024]
year_random_draw = rand(baseline_years)
    
toy_data_full_year = DataFrame()  
append!(toy_data_full_year, hourly_data[hourly_data.year .== year_random_draw, :])

day_start = rand(1:21)
toy_data = filter(row -> row.day >= day_start && row.day < day_start + 7, toy_data_full_year)
#toy_data = filter(row -> row.month <= 4, toy_data)

toy_data.total_demand_mwh = toy_data.residential_demand_mwh .+ toy_data.industrial_demand_mwh .+ toy_data.commercial_demand_mwh

# Add generation_mwh column (row-wise sum of selected generation types)
generation_cols = [:coal_gen_mwh, :combined_cycle_gen_mwh, :gas_turbine_gen_mwh, :vapor_turbine_gen_mwh, :cogeneration_gen_mwh, :diesel_gen_mwh, :nonrenewable_waste_gen_mwh,
                   :nuclear_gen_mwh, :conventional_hydro_gen_mwh, :run_of_river_hydro_gen_mwh, :pumped_hydro_gen_mwh, 
                   :solar_pv_gen_mwh, :solar_thermal_gen_mwh, :wind_gen_mwh, :other_renewable_gen_mwh, :renewable_waste_gen_mwh, :batteries_gen_mwh]

toy_data.generation_mwh = [sum(row) for row in eachrow(select(toy_data, generation_cols))]

# Add total_imports_mwh column
import_cols = [:imports_France_mwh, :imports_Portugal_mwh, :imports_Morocco_mwh]
toy_data.total_imports_mwh = [sum(row) for row in eachrow(select(toy_data, import_cols))]

# For the descriptive statistics (exclude gas_turbine and vapor_turbine as they are barely used in the model)
vars = [:spot_price_eur_mwh, :total_demand_mwh, :residential_demand_mwh, :commercial_demand_mwh, :industrial_demand_mwh, :generation_mwh,
        :coal_gen_mwh, :combined_cycle_gen_mwh, :cogeneration_gen_mwh, :diesel_gen_mwh, :nonrenewable_waste_gen_mwh, :nuclear_gen_mwh, 
        :conventional_hydro_gen_mwh, :run_of_river_hydro_gen_mwh, :pumped_hydro_gen_mwh, :solar_pv_gen_mwh, :solar_thermal_gen_mwh, :wind_gen_mwh, :other_renewable_gen_mwh, :renewable_waste_gen_mwh,
        :batteries_gen_mwh, :imports_France_mwh, :imports_Portugal_mwh, :imports_Morocco_mwh, 
        :solar_pv_cap_factor, :solar_thermal_cap_factor, :wind_cap_factor]

actual_descriptive_stats = describe(select(toy_data, vars), :min, :mean, :median, :max, :std)

Row,variable,min,mean,median,max,std
,Symbol,Float64,Float64,Float64,Float64,Float64
1,spot_price_eur_mwh,0.5,85.479,91.0,186.59,41.0515
2,total_demand_mwh,17.7193,27.9058,27.9073,39.8648,4.54621
3,residential_demand_mwh,7.05106,13.0535,12.9237,23.0024,3.32828
4,commercial_demand_mwh,5.17416,9.04904,8.99163,13.1317,1.82764
5,industrial_demand_mwh,4.55593,5.80329,5.83453,6.92973,0.387612
6,generation_mwh,18.1307,30.7177,31.1263,43.5761,5.05764
7,coal_gen_mwh,0.1835,0.473691,0.456292,1.26292,0.194284
8,combined_cycle_gen_mwh,1.05933,5.2443,4.60646,14.9481,2.85504
9,cogeneration_gen_mwh,0.806159,1.96536,2.05894,2.75562,0.453579


In [ ]:
results = dispatch_electricity_market(toy_data, fixed_data)

## We create a table to contrast model-solved results with actual stats

In [ ]:
# Create empty DataFrame for summary stats from model
summary_model_stats = DataFrame(
  variable = String[],
  min_model = Float64[],
  mean_model = Float64[],
  median_model = Float64[],
  max_model = Float64[],
  std_model = Float64[]
)

# List of active variable names from the results dictionary (exclude gas_turbine and vapor_turbine as they are barely used in the model)
variables = [
    "price", "total_demand", "residential_demand", "commercial_demand", "industrial_demand", "generation",
    "coal_gen", "combined_cycle_gen", "cogeneration_gen", "diesel_gen", "non_renewable_waste_gen", "nuclear_gen", 
    "conventional_hydro_gen", "run_of_river_hydro_gen", "pumped_hydro_gen", "solar_pv_gen", "solar_thermal_gen", "wind_gen", "other_renewable_gen", "renewable_waste_gen", 
    "battery_gen",
    "imports_FRA", "imports_POR", "imports_MOR",
    "share_renewable_gen"
]

# Populate the DataFrame with statistics from `results`
for var in variables
    values = results[var]  # access vector from dictionary using key

    push!(summary_model_stats, (
        variable = var,
        min_model = minimum(values),
        mean_model = mean(values),
        median_model = median(values),
        max_model = maximum(values),
        std_model = std(values)
    ))
end

# Trim actual descriptive stats to match number of model variables (22 before appending share_renewable_gen)
actual_stats_trimmed = actual_descriptive_stats[1:24, :]

# Add a new column for share of renewable generation to toy_data
renewable_cols = [:conventional_hydro_gen_mwh, :run_of_river_hydro_gen_mwh, :pumped_hydro_gen_mwh, 
    :solar_pv_gen_mwh, :solar_thermal_gen_mwh, :wind_gen_mwh, :other_renewable_gen_mwh, :renewable_waste_gen_mwh]
toy_data.share_renewable_gen = [
    total > 0 ? sum(row[renewable_cols]) / total : 0 
    for (row, total) in zip(eachrow(toy_data), toy_data.generation_mwh)
]

# Compute share_renewable_gen statistics from REAL data (toy_data)
real_share_stats = DataFrame(
    variable = "share_renewable_gen",
    min = minimum(toy_data.share_renewable_gen),
    mean = mean(toy_data.share_renewable_gen),
    median = median(toy_data.share_renewable_gen),
    max = maximum(toy_data.share_renewable_gen),
    std = std(toy_data.share_renewable_gen)
)

# Append this row to the actual stats
actual_stats_trimmed = vcat(actual_stats_trimmed, real_share_stats)

# Prepare actual stats for merging
actual_stats_merge = select(actual_stats_trimmed, Not("variable"))
new_names = names(actual_stats_merge) .=> (n -> n * "_real").(names(actual_stats_merge))
rename!(actual_stats_merge, new_names)

# Combine model and real stats
combined_stats = hcat(summary_model_stats, actual_stats_merge)

# Reorder columns for clarity
metrics = ["min", "mean", "median", "max", "std"]
ordered_cols = ["variable"]
for m in metrics
    push!(ordered_cols, m * "_real")
    push!(ordered_cols, m * "_model")
end

combined_stats = combined_stats[:, ordered_cols]

# Output
combined_stats

In [ ]:
# Filter the results dictionary to include only the specified keys, to being able to make the graphs
filtered_results = Dict(
    "residential_demand" => results["residential_demand"],
    "commercial_demand" => results["commercial_demand"],
    "industrial_demand" => results["industrial_demand"],
    "price" => results["price"],
    "coal_gen" => results["coal_gen"],
    "combined_cycle_gen" => results["combined_cycle_gen"],
    "gas_turbine_gen" => results["gas_turbine_gen"],
    "vapor_turbine_gen" => results["vapor_turbine_gen"],
    "cogeneration_gen" => results["cogeneration_gen"],
    "diesel_gen" => results["diesel_gen"],
    "non_renewable_waste_gen" => results["non_renewable_waste_gen"],
    "conventional_hydro_gen" => results["conventional_hydro_gen"],
    "run_of_river_hydro_gen" => results["run_of_river_hydro_gen"],
    "pumped_hydro_gen" => results["pumped_hydro_gen"],
    "nuclear_gen" => results["nuclear_gen"],
    "solar_pv_gen" => results["solar_pv_gen"],
    "solar_thermal_gen" => results["solar_thermal_gen"],
    "wind_gen" => results["wind_gen"],
    "other_renewable_gen" => results["other_renewable_gen"],
    "renewable_waste_gen" => results["renewable_waste_gen"],
    "battery_gen" => results["battery_gen"],
    "imports_FRA" => results["imports_FRA"],
    "imports_POR" => results["imports_POR"],
    "imports_MOR" => results["imports_MOR"],
    "exports_FRA" => results["exports_FRA"],
    "exports_POR" => results["exports_POR"],
    "exports_MOR" => results["exports_MOR"],
    "share_renewable_gen" => results["share_renewable_gen"],
    )

# Convert the filtered results dictionary into a DataFrame
results_df = DataFrame(filtered_results)

# Add a time column for better visualization
results_df[!, :time] = 1:size(results_df, 1)

In [ ]:
# Plot actual price vs model-solved price across time
plot(1:length(toy_data.spot_price_eur_mwh), toy_data.spot_price_eur_mwh, 
  label="Actual Price (€/MWh)", xlabel="Time (hours)", ylabel="Price (€/MWh)", legend=:topright)
plot!(results_df[!, :time], results_df[!, :price], label="Model-Solved Price (€/MWh)")

In [ ]:
# Determine the common y-axis range
y_min = min(minimum(toy_data.residential_demand_mwh), minimum(toy_data.industrial_demand_mwh))
y_max = max(maximum(toy_data.residential_demand_mwh), maximum(toy_data.industrial_demand_mwh))

# Plot actual vs model-solved residential demand
plot(1:length(toy_data.residential_demand_mwh), toy_data.residential_demand_mwh, 
    label="Actual residential demand (MWh)", xlabel="Time (hours)", ylabel="Demand (MWh)", legend=:topright, 
    title="Residential Demand", ylim=(y_min, y_max), yformatter = :plain)
plot!(results_df[!, :time], results_df[!, :residential_demand], label="Model-Solved residential demand (MWh)")

In [ ]:
#Plot actual vs model-solved commercial demand
plot(1:length(toy_data.commercial_demand_mwh), toy_data.commercial_demand_mwh, 
    label="Actual commercial demand (MWh)", xlabel="Time (hours)", ylabel="Demand (MWh)", legend=:topright, 
    title="commercial Demand", ylim=(y_min, y_max), yformatter = :plain)
plot!(results_df[!, :time], results_df[!, :commercial_demand], label="Model-Solved commercial demand (MWh)")

In [ ]:
# Plot actual vs model-solved industrial demand
plot(1:length(toy_data.industrial_demand_mwh), toy_data.industrial_demand_mwh, 
    label="Actual industrial demand (MWh)", xlabel="Time (hours)", ylabel="Demand (MWh)", legend=:topright, 
    title="Industrial Demand", ylim=(y_min, y_max), yformatter = :plain)
plot!(results_df[!, :time], results_df[!, :industrial_demand], label="Model-Solved industrial demand (MWh)")

In [ ]:
# Plot actual coal vs model-solved coal across time
plot(1:length(toy_data.coal_gen_mwh), toy_data.coal_gen_mwh, 
label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Coal generation (MWh)", legend=:topright)
plot!(results_df[!, :time], results_df[!, :coal_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual coal vs model-solved coal across time
plot(1:length(toy_data.cogeneration_gen_mwh), toy_data.cogeneration_gen_mwh, 
label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Cogeneration generation (MWh)", legend=:topright)
plot!(results_df[!, :time], results_df[!, :cogeneration_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual combined cycle generation vs model-solved combined cycle generation across time
plot(1:length(toy_data.combined_cycle_gen_mwh), toy_data.combined_cycle_gen_mwh, 
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Natural gas generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :combined_cycle_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual combined cycle generation vs model-solved combined cycle generation across time
plot(1:length(toy_data.nonrenewable_waste_gen_mwh), toy_data.nonrenewable_waste_gen_mwh, 
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Non-renewable waste generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :non_renewable_waste_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual nuclear vs model-solved nuclear across time
plot(1:length(toy_data.nuclear_gen_mwh), toy_data.nuclear_gen_mwh, 
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Nuclear generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :nuclear_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual hydro vs model-solved conventional hydro across time
plot(1:length(toy_data.conventional_hydro_gen_mwh), toy_data.conventional_hydro_gen_mwh,
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Conventional hydro generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :conventional_hydro_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual run of river hydro vs model-solved run of river hydro across time
plot(1:length(toy_data.run_of_river_hydro_gen_mwh), toy_data.run_of_river_hydro_gen_mwh,
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Run of river hydro generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :run_of_river_hydro_gen], label="Model-Solved generation (MWh)")


In [ ]:
# Plot actual pumped hydro vs model-solved pumped hydro across time with transparency for model line
plot(1:length(toy_data.pumped_hydro_gen_mwh), toy_data.pumped_hydro_gen_mwh,
    label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Pumped hydro generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :pumped_hydro_gen], label="Model-Solved generation (MWh)", alpha=0.5)

In [ ]:
# Plot actual solar_pv vs model-solved solar_pv across time
plot(1:length(toy_data.solar_pv_gen_mwh), toy_data.solar_pv_gen_mwh, 
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Solar PV generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :solar_pv_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual solar_thermal vs model-solved solar_thermal across time
plot(1:length(toy_data.solar_thermal_gen_mwh), toy_data.solar_thermal_gen_mwh, title="Solar thermal generation",
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Solar thermal generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :solar_thermal_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual wind vs model-solved wind across time
plot(1:length(toy_data.wind_gen_mwh), toy_data.wind_gen_mwh, title="Wind generation",
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Wind generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :wind_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual other renewable vs model-solved other renewable across time
plot(1:length(toy_data.other_renewable_gen_mwh), toy_data.other_renewable_gen_mwh, label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Other renewable generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :other_renewable_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual renewable waste vs model-solved renewable waste across time
plot(1:length(toy_data.renewable_waste_gen_mwh), toy_data.renewable_waste_gen_mwh, label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Renewable waste generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :renewable_waste_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual battery use vs model-solved battery use across time
plot(1:length(toy_data.batteries_gen_mwh), toy_data.batteries_gen_mwh, label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Batteries generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :battery_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual imports from France vs model-solved imports from France across time
plot(1:length(toy_data.imports_France_mwh), toy_data.imports_France_mwh, 
  label="Actual imports (MWh)", xlabel="Time (hours)", ylabel="Imports from France (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :imports_FRA], label="Model-Solved imports (MWh)")

In [ ]:
# Plot actual imports from Portugal vs model-solved imports from Portugal across time
plot(1:length(toy_data.imports_Portugal_mwh), toy_data.imports_Portugal_mwh, 
  label="Actual imports (MWh)", xlabel="Time (hours)", ylabel="Imports from Portugal (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :imports_POR], label="Model-Solved imports (MWh)")

In [ ]:
# Plot actual imports from Morocco vs model-solved imports from Morocco across time
plot(1:length(toy_data.imports_Morocco_mwh), toy_data.imports_Morocco_mwh, 
  label="Actual imports (MWh)", xlabel="Time (hours)", ylabel="Imports from Morocco (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :imports_MOR], label="Model-Solved imports (MWh)")

In [ ]:
# Plot actual exports to France vs model-solved exports to France across time
plot(1:length(toy_data.exports_France_mwh), toy_data.exports_France_mwh, 
  label="Actual exports (MWh)", xlabel="Time (hours)", ylabel="Exports to France (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :exports_FRA], label="Model-Solved exports (MWh)")

In [ ]:
# Plot actual exports from Portugal vs model-solved exports from Portugal across time
plot(1:length(toy_data.exports_Portugal_mwh), toy_data.exports_Portugal_mwh, 
  label="Actual exports (MWh)", xlabel="Time (hours)", ylabel="Exports to Portugal (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :exports_POR], label="Model-Solved exports (MWh)")

In [ ]:
# Plot actual exports from Morocco vs model-solved exports from Morocco across time
plot(1:length(toy_data.exports_Morocco_mwh), toy_data.exports_Morocco_mwh, 
  label="Actual exports (MWh)", xlabel="Time (hours)", ylabel="Exports to Morocco (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :exports_MOR], label="Model-Solved exports (MWh)")

In [ ]:
# Plot actual share of renewable generation vs model-solved share of renewable generation across time
plot(1:length(toy_data.share_renewable_gen), toy_data.share_renewable_gen, 
  label="Actual share of renewable generation", xlabel="Time (hours)", ylabel="Share of renewable generation", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :share_renewable_gen], label="Model-Solved share of renewable generation")   

# Add a line which is the mean share of renewable generation in the plot
mean_share = mean(results_df[!, :share_renewable_gen])
plot!(1:length(results_df[!, :time]), fill(mean_share, length(results_df[!, :time])), label="Mean Share", linestyle=:dash, color=:red)

In [ ]:
mean(toy_data.cost_coal_eur_mwh), mean(toy_data.cost_gas_eur_mwh), mean(toy_data.cost_diesel_eur_mwh), mean(toy_data.cost_uranium_eur_mwh)

In [ ]:
# Marginal cost of coal
63 * 0.82 + 8.55 + 16.34 / 0.36

In [ ]:
# Marginal cost of gas
63 * 0.49 + 2.29 + 46.54 / 0.57

In [ ]:
mean(toy_data.coal_gen_mwh), minimum(toy_data.coal_gen_mwh)